# Friend A - B20 DivNet division ranking (resumable)

This is the current 18 September task from the top of `PROMPT_FRIEND_A.md`. Dense fine-tuning and B5 TTA are cancelled. The notebook runs the public exp29 port over all 199 local videos, builds the pass-1 checkpoints, scores candidate divisions with the public DivNet model, and evaluates the fixed `top-k x threshold` grid with the official metric.

Expected duration on an RTX 5070 Ti is roughly 9-14 hours, mostly in pass 1. The exact public control is always evaluated before the DivNet variants. Nothing is submitted to Kaggle.

## Safe stop and resume

- Use **Interrupt Kernel**. Completed videos and DivNet batches remain reusable.
- Or create `C:\\Projects\\CellTracker\\outputs\\local_friend_a_b20_divnet\\STOP_REQUESTED` from another PowerShell window.
- Run All again to resume. Existing raw graphs, checkpoints, DivNet scores and pass-2 outputs are skipped.
- Optional before Jupyter: `$env:BIOHUB_B20_MAX_VIDEOS = "1"` for a short development run, or `$env:BIOHUB_B20_RUN = "0"` for preflight only.


In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
candidates = [start, start / '.github-push-worktree']
for parent in start.parents:
    candidates.extend([parent, parent / '.github-push-worktree'])
REPO_ROOT = next((p for p in candidates if (p / 'TEAM.md').is_file() and (p / 'tools' / 'public_local.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Repository root not found. Start Jupyter from C:\\Projects\\CellTracker.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from experiments.local_friend_a_b20_divnet.runner import (
    B20Config, clear_stop, compact_status, ensure_dependencies,
    preflight, request_stop, run_pipeline,
)
print('Repository:', REPO_ROOT)


In [ ]:
cfg = B20Config.from_environment(REPO_ROOT)
cfg.serialisable()


In [ ]:
# Downloads only missing public model packs. Re-running is safe.
if cfg.run_pipeline:
    ensure_dependencies(cfg)


In [ ]:
# A stale cooperative-stop marker is cleared only when a new Run All begins.
clear_stop(cfg)
check = preflight(cfg, gpu_smoke=cfg.run_pipeline)
check


The next cell first completes a one-video end-to-end smoke test. It then continues pass 1 one video at a time, checkpoints DivNet inside each video, and writes each pass-2 graph before moving on. A repeated Run All resumes instead of restarting.


In [ ]:
state = run_pipeline(cfg)
compact_status(cfg)


In [ ]:
status = compact_status(cfg)
result_path = cfg.output_dir / 'result.md'
if result_path.is_file():
    print(result_path.read_text(encoding='utf-8'))
print('Cooperative stop marker:', cfg.stop_file)
print('From another Python session you may call request_stop(cfg).')
